# S6E11 | Wide LM Stack + Part 9 + Shallow-Tree | LB 0.95300

**Our earlier versions** (all in the version history): v11 0.95291 (old wide stack + 20% shallow-tree, above every public notebook at the time), v10 0.95288, v9 GLM-Boost + Residual MLP Pool 0.95239.

This version scores **0.95300** on the public LB, level with the best public notebooks. It rebuilds our blend on the new Part 12 wide stack and re-checks every weight on the public LB and in cross-validation.

## What it does

1. **Backbone (0.95300).** goodpjw2008 rebuilt his wide stack with a second language-model member (Part 12, `wide_lm_stack_test.npy`; nested OOF AUC 0.952565 against 0.952504 for the old wide stack). It is mixed with his frozen Part 9 predictions as 65/35 logit ranks, the Fraud Signal Lab recipe by Malyshev Danil. This backbone alone scores 0.95300.
2. **Shallow-tree member at 10%.** megayak Per-Value + Digit TE Shallow-Tree Stack, also as logit ranks.
3. The mixture is ranked once more and written to `submission.csv`. Both goodpjw2008 arrays are checked against SHA-256 hashes, and the code warns if the megayak file has changed since this version was scored.

## How the weights were chosen

Eight public-LB probes on the same day, all as logit ranks:

| Blend | Public LB |
|---|---|
| backbone: 65/35 wide LM stack + Part 9 | 0.95300 |
| **backbone + shallow-tree 10% (this notebook)** | **0.95300** |
| backbone + shallow-tree 15% / 20% | 0.95299 / 0.95297 |
| backbone + Sovereign Fusion (amanatar) 15% / 30% | 0.95298 / 0.95296 |
| backbone + Part 10 (goodpjw2008) 20% | 0.95296 |
| backbone split 75/25 / 55/45 instead of 65/35 | 0.95299 / 0.95297 |

- **The backbone is a local optimum.** Every member we added cost 0.00001 to 0.00004, and moving its split either way cost too.
- **The 10% shallow-tree is the one addition that holds up in CV.** Adding megayak GLM-boost OOF at 10% to the nested OOF of the wide LM stack raises CV AUC by +0.00001; 20% and 30% add less. On the public LB it ties.
- **Why our old 20% no longer wins:** the second LM member already carries most of what the shallow-tree added to the old wide stack.
- **How to read the gaps:** the public LB uses about 20% of the test set, so 0.00001 is noise. The final ranking uses the other 80%, where the CV-backed choice is the safer one.

## Credits

- **goodpjw2008**, [S6E11 Part 12: Wide Stack + 2nd LM](https://www.kaggle.com/code/goodpjw2008/s6e11-part-12-wide-stack-2nd-lm-lb-0-95298) and the [s6e11-fraud-series-artifacts](https://www.kaggle.com/datasets/goodpjw2008/s6e11-fraud-series-artifacts) dataset: the wide LM stack, Part 9 and the OOF predictions used for the CV check.
- **Malyshev Danil**, [Fraud Signal Lab | GPT-2 and GLM Fusion](https://www.kaggle.com/code/malyshevdanil/fraud-signal-lab-gpt-2-and-glm-fusion): the 65/35 rank-logit backbone recipe.
- **megayak**, [S6E11 | Per-Value + Digit TE | Shallow-Tree Stack](https://www.kaggle.com/code/megayak/s6e11-per-value-digit-te-shallow-tree-stack): the shallow-tree predictions mixed in at 10%, and the member OOFs used for the CV check.
- **amanatar** ([Sovereign Fusion](https://www.kaggle.com/code/amanatar/s6e11-sovereign-fusion-solution)) and goodpjw2008 (Part 10): probed in the table above, not used in the final blend.
- The wide stack pools public predictions from Sadam Torres, alexchilton, Malyshev Danil, lazyyeou, megayak, Will, queryrusi, Vladimir Demidov, Nilanjan_Saha123, masayakawamata, Sameer Shaikh and goodpjw2008. See goodpjw2008 notebooks for where each model comes from.

If this helps, please upvote the source notebooks too.


In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from scipy.special import logit

# Frozen goodpjw2008 arrays (checked by SHA-256) and the megayak shallow-tree submission
HASHES = {
    'wide_lm_stack_test.npy': 'b37471e6c34a50715284e807d68e15e5f3ad033e72400904f27876b24041f8cc',
    'part9_test.npy': 'e899205391b8755e82a2ae6b06fb1e026ef29132e4bcdbfad96535d7f5608abd',
}
SHALLOW_SHA = '96abe3b112cdf794c56f4bbd8f7a70f5b5ebffd1c7be3436f636c59fb581c1a7'

inputs = Path('/kaggle/input')

def locate(name):
    paths = list(inputs.rglob(name))
    assert len(paths) == 1, (name, paths)
    return paths[0]

def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def logit_rank(v):
    return logit((rankdata(v) - 0.5) / len(v))

sample = pd.read_csv(locate('sample_submission.csv'))
arr = {}
for name, expected in HASHES.items():
    path = locate(name)
    assert sha(path) == expected, name
    arr[name] = np.load(path, allow_pickle=False).ravel()

shallow_path = locate('submission.csv')
if sha(shallow_path) != SHALLOW_SHA:
    print('WARNING: the megayak shallow-tree submission has changed since this version was scored')
shallow = pd.read_csv(shallow_path)
assert (shallow['id'].values == sample['id'].values).all()

# Step 1: the 0.95300 backbone, 65/35 logit ranks of the wide LM stack (Part 12) and Part 9
backbone = 0.65 * logit_rank(arr['wide_lm_stack_test.npy']) + 0.35 * logit_rank(arr['part9_test.npy'])
# Step 2: mix in the shallow-tree stack at 10%, also as logit ranks
score = 0.90 * logit_rank(backbone) + 0.10 * logit_rank(shallow['is_fraud'].values)

submission = sample[['id']].copy()
submission['is_fraud'] = (rankdata(score) - 0.5) / len(score)
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission.shape)
print('SHA256', sha(Path('/kaggle/working/submission.csv')))


In [ ]:
import pandas as pd
sub = pd.read_csv('/kaggle/working/submission.csv')
print(sub.shape)
print(sub.head())
print(sub['is_fraud'].describe())